# Pydantic Library

Pydantic is a Python library used for runtime data validation, parsing, and type enforcement. In AI and Large Language Model (LLM) engineering, it acts as a strict "security guard" that ensures messy, unstructured, or dynamically generated data matches an exact shape before your code or model processes it.


For example in below code, it provides
### Guaranteed Type Safety:
The profile.age variable is guaranteed to be a Python int, and profile.skills is guaranteed to be a standard Python list. Your code won't crash trying to iterate over a comma-separated string.

### Auto-Generated Prompts:
The description fields inside Field() are sent directly to the LLM behind the scenes, helping the model understand exactly what you expect in each field without you needing to write explicit instructions in the system prompt.

In [1]:
import os
from getpass import getpass

# Securely prompt for the OpenAI API Key
api_key = getpass("Enter your OpenAI API Key: ")
os.environ["OPENAI_API_KEY"] = api_key
print("OPENAI_API_KEY has been set as an environment variable.")

Enter your OpenAI API Key: ··········
OPENAI_API_KEY has been set as an environment variable.


In [2]:
import os
from openai import OpenAI
from pydantic import BaseModel, Field


# 1. Initialize the OpenAI client
# Ensure your OPENAI_API_KEY environment variable is set
client = OpenAI()

# 2. Unstructured raw text from which to extract data
unstructured_text = (
    "Hey, I'm Alex. I moved to New York last year and I love it here. "
    "I'm 28 years old and work as a software engineer. "
    "Most of my days are spent working with Python, Docker, and AWS."
)

# 3. Define your desired output structure using Pydantic
class UserProfile(BaseModel):
    name: str
    age: int = Field(description="The user's age in years")
    location: str
    skills: list[str] = Field(description="List of technical or professional skills")


# 4. Call the LLM with the Pydantic schema enforced
completion = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Extract the user profile information from the text."},
        {"role": "user", "content": unstructured_text},
    ],
    response_format=UserProfile,  # Enforce the Pydantic schema here
)

# 5. Access the parsed, type-safe data
profile: UserProfile = completion.choices[0].message.parsed

# Print the validated data
print("Parsed Object Type:", type(profile))
print("Name:", profile.name)
print("Age:", profile.age)
print("Location:", profile.location)
print("Skills:", profile.skills)

# Convert back to standard JSON/dict if needed
print("\nJSON Output:")
print(profile.model_dump_json(indent=2))


Parsed Object Type: <class '__main__.UserProfile'>
Name: Alex
Age: 28
Location: New York
Skills: ['Python', 'Docker', 'AWS']

JSON Output:
{
  "name": "Alex",
  "age": 28,
  "location": "New York",
  "skills": [
    "Python",
    "Docker",
    "AWS"
  ]
}
